# 01 — Data Loading & Merge

**Goal of this notebook**: turn the raw Figshare `.mat` files and Br35H "no tumor"
images into a single `metadata.csv` — the source of truth every later notebook reads
from. We deliberately do *not* load full image arrays into memory here; only paths
and labels, so this step stays fast and safe to re-run.

**Why this step matters for data integrity**: the Figshare files store the patient ID
(`cjdata.PID`) as a MATLAB string, which in the HDF5-based `.mat` format is an array of
character codes, not a plain scalar — `src/data_utils.py` handles this decoding
correctly (`_decode_matlab_string`), verified against a synthetic file matching the real
structure in `tests/test_data_utils.py`.

**Prerequisite**: raw data must already be downloaded per `data/README.md` before
running this notebook — it is not fetched automatically.

In [ ]:
import sys

sys.path.insert(0, "..")

from pathlib import Path

from src.data_utils import build_metadata

FIGSHARE_DIR = Path("../data/raw/figshare_mat")
BR35H_DIR = Path("../data/raw/br35h_no_tumor")
OUTPUT_CSV = Path("../data/processed/metadata.csv")

## Build the unified metadata table

This scans both raw sources and writes `metadata.csv` with columns:
`image_path, label, patient_id, source_dataset`.

In [ ]:
metadata = build_metadata(FIGSHARE_DIR, BR35H_DIR, OUTPUT_CSV)
print(f"Total images: {len(metadata)}")
metadata.head()

## Sanity checks before moving on

Quick checks that the merge did what we expect — class presence, patient ID
uniqueness pattern (Figshare patients should repeat across slices, Br35H should not).

In [ ]:
print("Class counts:")
print(metadata["label"].value_counts())

print("\nSource dataset counts:")
print(metadata["source_dataset"].value_counts())

print("\nSlices per Figshare patient (should mostly be >1):")
figshare_only = metadata[metadata["source_dataset"] == "figshare"]
print(figshare_only.groupby("patient_id").size().describe())

print("\nBr35H images per synthetic patient_id (should all be exactly 1):")
br35h_only = metadata[metadata["source_dataset"] == "br35h"]
assert (
    br35h_only.groupby("patient_id").size() == 1
).all(), "Br35H patient IDs should be unique per image"
print("OK — every Br35H image has its own unique group.")

Next notebook: **02_eda.ipynb** — explore class balance, patient distribution, and
sample images before we touch any modeling.